In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

data= pd.read_csv("pos_tags.csv", nrows=2000)
print(data.head())


sentences=[]
temp=[]

for _, row in data.iterrows():
    temp.append((str(row["word"]), str(row["tag"])))
    if len(temp) == 20:
        sentences.append(temp)
        temp = []

train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)

vocab= set()
tag_set= set()

for sentence in train_data:
    for word, tag in sentence:
        vocab.add(word.lower())
        tag_set.add(tag)

vocab = sorted(list(vocab))
tags = sorted(list(tag_set))

word_to_idx = {w:i for i, w in enumerate(vocab)}
tag_to_idx = {t: i for i, t in enumerate(tags)}
idx_to_tag = {i: t for t, i in tag_to_idx.items()}

V = len(vocab)
T = len(tags)

print("\nVocabulary Size:",V)
print("\nNumber of Tags:",T)

initial = np.ones(T)
transition = np.ones((T, T))
emission = np.ones((T, V))

for sentence in train_data:
    first_tag = sentence[0][1]
    initial[tag_to_idx[first_tag]] += 1

    for i, (word, tag) in enumerate(sentence):
        word = word.lower()
        tag_idx = tag_to_idx[tag]

        if word in word_to_idx:
            emission[tag_idx, word_to_idx[word]] += 1

        if i > 0:
            prev_tag = sentence[i-1][1]
            prev_idx = tag_to_idx[prev_tag]
            transition[prev_idx, tag_idx] += 1

initial = initial / initial.sum()
transition = transition / transition.sum(axis=1, keepdims=True)
emission = emission / emission.sum(axis=1, keepdims=True)

log_initial = np.log(initial)
log_transition = np.log(transition)
log_emission = np.log(emission)

def viterbi(sentence):
    n = len(sentence)
    dp = np.full((T, n), -np.inf)
    backpointer = np.zeros((T, n), dtype=int)

    word = sentence[0].lower()
    if word in word_to_idx:
        emit = log_emission[:, word_to_idx[word]]
    else:
        emit = np.log(np.ones(T) * 1e-10)
    dp[:, 0] = log_initial + emit

    for i in range(1, n):
        word = sentence[i].lower()
        if word in word_to_idx:
            emit = log_emission[:, word_to_idx[word]]
        else:
            emit = np.log(np.ones(T) * 1e-10)
        scores = dp[:, i-1][:, None] + log_transition
        backpointer[:, i] = np.argmax(scores, axis=0)
        dp[:, i] = np.max(scores, axis=0) + emit

    best_last = np.argmax(dp[:, -1])
    best_path = [best_last]
    for i in range(n-1, 0, -1):
        best_last = backpointer[best_last, i]
        best_path.append(best_last)
    best_path.reverse()
    return [idx_to_tag[i] for i in best_path]

y_true = []
y_pred = []
for sentence in test_data:
    words = [w for w, t in sentence]
    true_tags = [t for w, t in sentence]
    pred_tags = viterbi(words)
    y_true.extend(true_tags)
    y_pred.extend(pred_tags)

print("\n\nAccuracy:", accuracy_score(y_true, y_pred))
print("\n\nClassification Report:\n")
print(classification_report(y_true, y_pred))

unseen_sentences = [
    "Artificial intelligence improves healthcare systems",
    "Students are learning machine learning",
    "The dog barked loudly",
    "She writes beautiful poems",
    "Cloud computing enables scalable applications"
]

for sent in unseen_sentences:
    words = sent.split()
    tags_pred = viterbi(words)

    print("\n\nSentence:", sent)
    print("\n\nPredicted Tags:")

    for w, t in zip(words, tags_pred):
        print(f"{w:15s} --> {t}")


   sentence_id    word  tag
0            0      aa   NN
1            1     aaa   NN
2            2     aah   NN
3            3   aahed  VBN
4            4  aahing  VBG

Vocabulary Size: 1600

Number of Tags: 9


Accuracy: 0.565


Classification Report:

              precision    recall  f1-score   support

          JJ       0.00      0.00      0.00        45
          NN       0.56      1.00      0.72       226
         NNS       0.00      0.00      0.00        64
          RB       0.00      0.00      0.00        14
          VB       0.00      0.00      0.00         5
         VBD       0.00      0.00      0.00         2
         VBG       0.00      0.00      0.00        22
         VBN       0.00      0.00      0.00        22

    accuracy                           0.56       400
   macro avg       0.07      0.12      0.09       400
weighted avg       0.32      0.56      0.41       400



Sentence: Artificial intelligence improves healthcare systems


Predicted Tags:
Artificial   

D:\ANACONDA\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\ANACONDA\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\ANACONDA\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
